# Метрики OCR: GOT-OCR2.0 (+ опционально MinerU)

Ноутбук для **Google Colab** или **локально**: прогон **`ucaslcl/GOT-OCR2_0`**, метрики **CER / Accuracy / Final Score**; в конце — опциональный вызов **`scripts/mineru_image_benchmark.py`**.

---

## Google Colab — выполняйте ячейки сверху вниз

1. **Runtime → Change runtime type → GPU** (желательно; иначе GOT-OCR2 будет на CPU очень медленно).
2. **Шаг 2** — первая кодовая ячейка: в Colab клонирует репозиторий в `/content/OCR-Analyze`, переходит в корень, `pip install -r notes/requirements-ocr-notebook-colab.txt`.  
   - Свой форк: **`export OCR_ANALYZE_GIT_URL=...`** перед ячейкой или правка `GIT_URL` внутри.  
   - Другой каталог клона: **`export OCR_ANALYZE_COLAB_DIR=/content/мой-путь`**.  
   - Если Hugging Face из Colab недоступен — см. [документацию HF](https://huggingface.co/docs/huggingface_hub/guides/download#faster-downloads) (зеркала, `HF_TOKEN`).
3. **Шаг 3** — следующая ячейка: `REPO_ROOT`, список PNG и найденных эталонов.
4. **Шаги 4–5** — функции, затем загрузка GOT-OCR2 (долго при первом запуске).
5. **Шаг 6** — прогон по изображениям и таблица по файлам.
6. **Шаг 7** — сводная статистика и файлы в `output/got_ocr_notebook/`.
7. **Шаг 8 (опционально)** — MinerU; без установленного `mineru` ячейка просто пропускается.

В коде шаги дублируются комментариями **`# Шаг N`** в начале соответствующих ячеек — можно идти сверху вниз без карты.

**Локально:** шаг 2 не клонирует; откройте ноутбук из корня репозитория или задайте **`REPO_ROOT_OVERRIDE`** в ячейке путей.

---

## Поля метрик (кратко)

| Поле | Откуда |
|------|--------|
| **Model** | `MODEL_ID` + `OCR_TYPE` |
| **Final Score** | `100 × (1 − CER)` (микро по конкатенации) |
| **Accuracy** / **CER** | `jiwer`, после `NORMALIZE_MODE` |
| **Unit Test Rate** | порог CER + опционально `ocr_unit_expectations.json` |
| **Токены** | `tokenizer.encode` по выводу |
| **Скорость** | среднее время на изображение в сводке |

**Эталоны:** `stem.ref.txt` → `stem.ref.md` → `stem.txt` → `stem.md`. Без эталона CER не считается; пишется черновик `stem.ref.txt` для правки вручную.

Зависимости GOT: файл **`notes/requirements-ocr-notebook-colab.txt`** (в т.ч. **verovio** для remote code модели).

In [ ]:
# Шаг 2 — Colab: клон репозитория + pip install. Локально: только pip (клон пропускается).

from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path


def in_colab() -> bool:
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


GIT_URL = os.environ.get(
    "OCR_ANALYZE_GIT_URL",
    "https://github.com/developer-mixa/OCR-Analyze.git",
)
REPO_DIR = Path(os.environ.get("OCR_ANALYZE_COLAB_DIR", "/content/OCR-Analyze"))

if in_colab():
    if not (REPO_DIR / "scripts").is_dir():
        print("Клонирую", GIT_URL, "→", REPO_DIR)
        subprocess.check_call(["git", "clone", "--depth", "1", GIT_URL, str(REPO_DIR)])
    os.chdir(REPO_DIR)
    print("Рабочий каталог:", Path.cwd().resolve())
else:
    print("Не Colab — клон не выполняется. cwd:", Path.cwd().resolve())

req = Path("notes/requirements-ocr-notebook-colab.txt")
if not req.is_file():
    print("WARN: нет", req, "— установите зависимости вручную (jiwer, pandas, …).")
else:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(req)],
        cwd=str(Path.cwd()),
    )
    print("OK: pip install -r", req)

print("\nДальше по порядку: Шаг 3 (пути и проверка данных) → Шаг 4 → Шаг 5 → Шаг 6 → Шаг 7 → опционально Шаг 8 (MinerU).")
print("После смены major-версий pip перезапустите runtime (Runtime → Restart session), если импорты ломаются.")

In [ ]:
from __future__ import annotations

# Шаг 3 — корень репозитория, каталоги, проверка PNG и эталонов.

import json
import re
import time
import unicodedata
from pathlib import Path

import jiwer
import pandas as pd
from IPython.display import HTML, display
from tqdm.auto import tqdm

# Если авто-поиск не найдёт репо — задайте абсолютный путь вручную и перезапустите ячейку.
REPO_ROOT_OVERRIDE: Path | None = None


def find_repo_root() -> Path:
    """Каталог с notes/ и scripts/ — от cwd вверх и среди прямых подпапок cwd."""
    if REPO_ROOT_OVERRIDE is not None:
        p = REPO_ROOT_OVERRIDE.expanduser().resolve()
        if (p / "notes").is_dir() and (p / "scripts").is_dir():
            return p
        print("WARN: REPO_ROOT_OVERRIDE задан, но нет notes/ и scripts/ — игнорируем.")
    cwd = Path.cwd().resolve()
    starts: list[Path] = []
    if cwd.name == "notes":
        starts.append(cwd.parent)
    starts.append(cwd)
    try:
        subdirs = sorted([p for p in cwd.iterdir() if p.is_dir()], key=lambda x: x.name.lower())
        starts.extend(subdirs)
    except OSError:
        pass
    seen: set[Path] = set()
    ordered: list[Path] = []
    for s in starts:
        s = s.resolve()
        if s not in seen:
            seen.add(s)
            ordered.append(s)
    for start in ordered:
        for cand in [start, *start.parents]:
            try:
                if (cand / "notes").is_dir() and (cand / "scripts").is_dir():
                    return cand
            except OSError:
                continue
    return cwd


REPO_ROOT = find_repo_root()

INPUT_DIR = REPO_ROOT / "input" / "data" / "1"
OUTPUT_DIR = REPO_ROOT / "output" / "got_ocr_notebook"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Опционально: JSON с проверками вида {"test1.png": {"must_contain": ["##", "Таблица"]}}
UNIT_TESTS_JSON = REPO_ROOT / "input" / "data" / "1" / "ocr_unit_expectations.json"

MODEL_ID = "ucaslcl/GOT-OCR2_0"
# Режимы из README GOT-OCR2: "ocr" | "format" | при необходимости расширьте вызов в infer_one
OCR_TYPE = "ocr"

NORMALIZE_MODE = "nfkc_ws"  # как task03 по умолчанию
CER_PASS_THRESHOLD = 0.05
INTERNAL_PARSER_LABEL = "none"

print("REPO_ROOT =", REPO_ROOT.resolve())
print("INPUT_DIR =", INPUT_DIR.resolve(), "exists:", INPUT_DIR.is_dir())

# Шаг 3: быстрая проверка входных данных
_png = sorted(INPUT_DIR.glob("*.png")) if INPUT_DIR.is_dir() else []
print(f"\nPNG в INPUT_DIR: {len(_png)}")
for p in _png:
    stem = p.stem
    refs = [n for n in (f"{stem}.ref.txt", f"{stem}.ref.md", f"{stem}.txt", f"{stem}.md") if (INPUT_DIR / n).is_file()]
    tag = f" эталон: {', '.join(refs)}" if refs else " эталона нет (после прогона может появиться черновик .ref.txt)"
    print(" ", p.name, "—", tag)
if not _png:
    print("  → Добавьте PNG в input/data/1 или проверьте clone / REPO_ROOT.")

In [ ]:
# Шаг 4 — вспомогательные функции (нормализация, CER, эталоны, dtype для шага 5).

def normalize_text(s: str, mode: str) -> str:
    if mode == "none":
        return s
    t = s
    if mode in ("nfkc", "nfkc_ws", "nfkc_ws_lower"):
        t = unicodedata.normalize("NFKC", t)
    if mode in ("nfkc_ws", "nfkc_ws_lower", "ws", "ws_lower"):
        t = re.sub(r"\s+", " ", t).strip()
    if mode in ("nfkc_ws_lower", "ws_lower"):
        t = t.lower()
    return t


def char_error_rate(ref: str, hyp: str) -> float:
    if not ref and not hyp:
        return 0.0
    if not ref:
        return 1.0
    return float(jiwer.cer(ref, hyp))


def load_reference_for_image(img_path: Path) -> str | None:
    """Эталон: stem.ref.txt, stem.ref.md, stem.txt, stem.md. Пустой/пробельный файл = нет эталона."""
    stem = img_path.stem
    for name in (f"{stem}.ref.txt", f"{stem}.ref.md", f"{stem}.txt", f"{stem}.md"):
        p = img_path.parent / name
        if p.is_file():
            text = p.read_text(encoding="utf-8")
            if text.strip():
                return text
    return None


def load_unit_expectations(path: Path) -> dict:
    if not path.is_file():
        return {}
    return json.loads(path.read_text(encoding="utf-8"))


def run_unit_checks(hyp_raw: str, rules: dict | None) -> tuple[bool, list[str]]:
    if not rules:
        return True, []
    fails: list[str] = []
    for sub in rules.get("must_contain") or []:
        if sub not in hyp_raw:
            fails.append(f"missing substring: {sub!r}")
    for sub in rules.get("must_not_contain") or []:
        if sub in hyp_raw:
            fails.append(f"forbidden substring present: {sub!r}")
    return (len(fails) == 0), fails


def pick_torch_dtype():
    import torch

    if not torch.cuda.is_available():
        return torch.float32, "cpu"
    major, _ = torch.cuda.get_device_capability()
    if major >= 8:
        return torch.bfloat16, "cuda"
    return torch.float16, "cuda"

In [ ]:
# Шаг 5 — загрузка модели GOT-OCR2.0 (долго при первом запуске: скачивание весов).

import torch
import transformers
from transformers import AutoModel, AutoTokenizer

print("transformers", transformers.__version__, "| torch", torch.__version__)
dtype, device_str = pick_torch_dtype()
print(f"device={device_str}, dtype={dtype}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

load_kw = dict(
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    use_safetensors=True,
    pad_token_id=tokenizer.eos_token_id,
)
if device_str == "cuda":
    load_kw["device_map"] = "cuda"

model = AutoModel.from_pretrained(MODEL_ID, **load_kw)
model.eval()
if device_str == "cpu":
    model = model.to(torch.float32)

print("OK: модель загружена.")

In [ ]:
# Шаг 5 (продолжение) — обёртка вызова model.chat.

def infer_one(image_path: Path) -> tuple[str, float]:
    t0 = time.perf_counter()
    res = model.chat(tokenizer, str(image_path), ocr_type=OCR_TYPE)
    elapsed = time.perf_counter() - t0
    hyp = res if isinstance(res, str) else str(res)
    return hyp, elapsed


def model_label() -> str:
    return f"{MODEL_ID} | chat ocr_type={OCR_TYPE} | low_cpu_mem_usage device_map=cuda(if cuda)"

## Шаг 6 — прогон по изображениям

Если каталога `input/data/1` нет — создайте его и добавьте PNG и (по желанию) эталоны `*.ref.txt` / `*.ref.md`.

Опционально — `input/data/1/ocr_unit_expectations.json` для **Unit Test Rate**:

```json
{ "page1.png": { "must_contain": ["#"] } }
```


In [ ]:
# Шаг 6 — цикл по PNG, запись гипотез и черновиков эталонов.

expectations = load_unit_expectations(UNIT_TESTS_JSON)

image_paths = sorted(INPUT_DIR.glob("*.png")) if INPUT_DIR.is_dir() else []
if not image_paths:
    print("Нет PNG в", INPUT_DIR, "— добавьте файлы для теста.")

rows: list[dict] = []
corpus_ref_parts: list[str] = []
corpus_hyp_parts: list[str] = []

for img_path in tqdm(image_paths, desc="GOT-OCR2"):
    ref_raw = load_reference_for_image(img_path)
    had_ref = bool(ref_raw and ref_raw.strip())
    try:
        hyp_raw, sec = infer_one(img_path)
    except Exception as e:
        rows.append(
            {
                "file": img_path.name,
                "error": repr(e),
                "elapsed_sec": None,
                "CER": None,
                "char_accuracy": None,
                "ref_chars": len(normalize_text(ref_raw, NORMALIZE_MODE)) if had_ref else None,
                "hyp_chars": None,
                "hyp_tokens": None,
                "cer_pass": None,
                "unit_ok": None,
                "unit_fails": None,
                "ref_status": None,
            }
        )
        continue

    out_md = OUTPUT_DIR / f"{img_path.stem}_hypothesis.md"
    out_md.write_text(hyp_raw, encoding="utf-8")

    ref_status = "эталон из файла" if had_ref else None
    if not had_ref and hyp_raw.strip():
        draft_ref = img_path.parent / f"{img_path.stem}.ref.txt"
        draft_ref.write_text(hyp_raw, encoding="utf-8")
        ref_status = f"черновик эталона записан (правьте и перезапустите): {draft_ref.name}"
    elif not had_ref:
        ref_status = "нет эталона, вывод OCR пуст — черновик не создан"

    ref_n = normalize_text(ref_raw or "", NORMALIZE_MODE)
    hyp_n = normalize_text(hyp_raw, NORMALIZE_MODE)
    cer = char_error_rate(ref_n, hyp_n) if had_ref else None
    acc = max(0.0, min(1.0, 1.0 - cer)) if cer is not None else None

    if had_ref:
        corpus_ref_parts.append(ref_n)
        corpus_hyp_parts.append(hyp_n)

    tok_len = None
    try:
        tok_len = len(tokenizer.encode(hyp_raw, add_special_tokens=False))
    except Exception:
        pass

    rules = expectations.get(img_path.name)
    u_ok, u_fails = run_unit_checks(hyp_raw, rules)
    cer_pass = (cer is not None and cer <= CER_PASS_THRESHOLD) if had_ref else None

    combined_pass = None
    if had_ref:
        combined_pass = cer_pass and u_ok
    elif rules:
        combined_pass = u_ok

    rows.append(
        {
            "file": img_path.name,
            "error": None,
            "elapsed_sec": round(sec, 4),
            "CER": round(cer, 6) if cer is not None else None,
            "char_accuracy": round(acc, 6) if acc is not None else None,
            "ref_chars": len(ref_n) if had_ref else None,
            "ref_status": ref_status,
            "hyp_chars": len(hyp_n),
            "hyp_tokens": tok_len,
            "cer_pass": cer_pass,
            "unit_ok": u_ok,
            "unit_fails": "; ".join(u_fails) if u_fails else None,
            "combined_pass": combined_pass,
        }
    )

df = pd.DataFrame(rows)
display(HTML("<h3>По файлам</h3>"))
if not df.empty:
    display(df.style.hide(axis="index").format(
        {"elapsed_sec": "{:.4f}", "CER": "{:.6f}"},
        na_rep="—",
    ))
else:
    display(df)

## Шаг 7 — сводная статистика (GOT)

Следующая ячейка (код) считает микро-CER, **Unit Test Rate**, сохраняет **`output/got_ocr_notebook/got_metrics_summary.json`** и **`got_per_file.csv`**. Выполняйте **после успешного шага 6**.

In [ ]:
# Шаг 7 — микро-CER, сводная строка, экспорт JSON/CSV.

# Микро-CER: одна пара больших строк (конкатенация с \n), эквивалентно сумме правок / длине эталона
micro_cer = (
    char_error_rate("\n".join(corpus_ref_parts), "\n".join(corpus_hyp_parts))
    if corpus_ref_parts
    else None
)
micro_acc = max(0.0, min(1.0, 1.0 - micro_cer)) if micro_cer is not None else None
final_score = round(100.0 * (1.0 - micro_cer), 2) if micro_cer is not None else None

evaluable = df["combined_pass"].notna() if not df.empty else pd.Series(dtype=bool)
unit_test_rate = float(df.loc[evaluable, "combined_pass"].mean()) if evaluable.any() else None

total_hyp_tokens = int(df["hyp_tokens"].fillna(0).sum()) if not df.empty else 0
mean_elapsed = float(df["elapsed_sec"].dropna().mean()) if not df.empty and df["elapsed_sec"].notna().any() else None

draft_ref_writes = 0
if not df.empty and "ref_status" in df.columns:
    draft_ref_writes = int(
        df["ref_status"].fillna("").str.contains("черновик эталона записан", regex=False).sum()
    )

comments = (
    f"model={MODEL_ID} ocr_type={OCR_TYPE}; device={device_str}; dtype={dtype}; "
    f"n_images={len(image_paths)}; with_ref_for_cer={df['CER'].notna().sum() if not df.empty else 0}; "
    f"draft_ref_txt_written={draft_ref_writes} (без эталона до прогона: CER не считался, записан черновик .ref.txt); "
    f"mean_elapsed_s={mean_elapsed}; "
    f"CER_PASS_THRESHOLD={CER_PASS_THRESHOLD}; normalize={NORMALIZE_MODE}; "
    f"expectations_file={UNIT_TESTS_JSON.name}"
)

summary_row = {
    "Model": model_label(),
    "Final Score": final_score,
    "Accuracy": round(micro_acc, 6) if micro_acc is not None else None,
    "CER": round(micro_cer, 6) if micro_cer is not None else None,
    "Unit Test Rate": round(unit_test_rate, 4) if unit_test_rate is not None else None,
    "Внутренний парсер": INTERNAL_PARSER_LABEL,
    "Токены": {
        "output_tokens_sum_hypothesis": total_hyp_tokens,
        "note": "сумма tokenizer.encode по сырому выводу модели по файлам",
    },
    "Доп. Комментарии": comments,
    "Скорость": {
        "mean_elapsed_sec_per_image": mean_elapsed,
        "output_dir": str(OUTPUT_DIR),
    },
}

summary_path = OUTPUT_DIR / "got_metrics_summary.json"
summary_path.write_text(json.dumps(summary_row, ensure_ascii=False, indent=2), encoding="utf-8")

sum_df = pd.DataFrame([summary_row])
display(HTML("<h3>Сводная строка (экспорт в JSON)</h3>"))
display(HTML(f"<p>Сохранено: <code>{summary_path}</code></p>"))

# Плоское отображение для копирования в таблицу
flat = {
    "Model": summary_row["Model"],
    "Final Score": summary_row["Final Score"],
    "Accuracy": summary_row["Accuracy"],
    "CER": summary_row["CER"],
    "Unit Test Rate": summary_row["Unit Test Rate"],
    "Внутренний парсер": summary_row["Внутренний парсер"],
    "Токены (sum)": summary_row["Токены"]["output_tokens_sum_hypothesis"],
    "Доп. Комментарии": summary_row["Доп. Комментарии"],
}
display(pd.DataFrame([flat]).style.hide(axis="index"))

csv_per_file = OUTPUT_DIR / "got_per_file.csv"
if not df.empty:
    df.to_csv(csv_per_file, index=False)
    print("CSV по файлам:", csv_per_file)

## Шаг 8 (опционально) — MinerU, `scripts/mineru_image_benchmark.py`

Прогон [MinerU](https://opendatalab.github.io/MinerU/) по тем же PNG, сравнение с эталонами `*.ref.txt` / `*.ref.md` рядом с файлами. Артефакты: `output/mineru_benchmark/hypotheses/mineru/*.md`, `mineru_runs.jsonl`, `mineru_summaries.json`.

**Нужно:** отдельно установить MinerU и **`mineru`** в PATH, `pip install jiwer` (если ещё не стоит из шага 2). В Colab обычно **GPU**; при проблемах с Hugging Face: `export MINERU_MODEL_SOURCE=modelscope`.

Ячейка с кодом ниже **пропускает** запуск, если `mineru` не найден (чтобы ноутбук не падал без MinerU).

In [ ]:
# Шаг 8 (опционально) — MinerU CLI. Пропуск, если mineru не в PATH.

import os
import shutil
import subprocess
import sys
from pathlib import Path

if "REPO_ROOT" not in globals() or "INPUT_DIR" not in globals():
    raise RuntimeError("Сначала выполните ячейки выше: должны быть определены REPO_ROOT и INPUT_DIR.")

mineru_bin = os.environ.get("MINERU_BIN", "mineru")
script = REPO_ROOT / "scripts" / "mineru_image_benchmark.py"
mineru_out = REPO_ROOT / "output" / "mineru_benchmark"

if not script.is_file():
    print("Пропуск: нет файла", script)
elif shutil.which(mineru_bin) is None and not Path(mineru_bin).is_file():
    print("Пропуск: не найден", repr(mineru_bin), "в PATH (установите MinerU и перезапустите ячейку).")
    print("Документация: https://opendatalab.github.io/MinerU/")
else:
    cmd = [
        sys.executable,
        str(script),
        "--input-dir",
        str(INPUT_DIR),
        "--output-dir",
        str(mineru_out),
        "--backend",
        os.environ.get("MINERU_BACKEND", "pipeline"),
        "--lang",
        os.environ.get("MINERU_LANG", "cyrillic"),
    ]
    if os.environ.get("MINERU_METHOD"):
        cmd.extend(["--method", os.environ["MINERU_METHOD"]])
    if mineru_bin != "mineru":
        cmd.extend(["--mineru-bin", mineru_bin])

    print("Команда:", " ".join(cmd))
    proc = subprocess.run(cmd, cwd=str(REPO_ROOT))
    print("Код возврата:", proc.returncode)

    summ = mineru_out / "mineru_summaries.json"
    if summ.is_file():
        print("\n--- mineru_summaries.json ---\n")
        print(summ.read_text(encoding="utf-8"))